# 测试

In [1]:
import set_env

/media/pc/data/lxw/ai/tvm-book


In [2]:
# import torch
# from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT).eval()
# shape = 1, 3, 224, 224
# trace = torch.jit.trace(model, torch.rand(shape).float()).eval()
# torch.jit.save(trace, ".temp/mobilenet_v2.pt")

In [3]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
from tvm import relay

model = resnet18(weights=ResNet18_Weights.DEFAULT).eval()
shape = 1, 3, 224, 224
trace = torch.jit.trace(model, torch.rand(shape).float()).eval()
torch.jit.save(trace, ".temp/resnet18.pt")

In [4]:
mod, params = relay.frontend.pytorch.from_pytorch(trace, [("x", shape)], use_parser_friendly_name=True)

In [5]:
import tvm

with tvm.transform.PassContext(opt_level=3):
    mod = relay.quantize.prerequisite_optimize(mod, params)
print(mod["main"])

fn (%x: Tensor[(1, 3, 224, 224), float32] /* ty=Tensor[(1, 3, 224, 224), float32] span=aten___convolution_0_x:0:0 */) -> Tensor[(1, 1000), float32] {
  %0 = nn.conv2d(%x, meta[relay.Constant][0] /* ty=Tensor[(64, 3, 7, 7), float32] */, strides=[2, 2], padding=[3, 3, 3, 3], channels=64, kernel_size=[7, 7]) /* ty=Tensor[(1, 64, 112, 112), float32] */;
  %1 = add(%0, meta[relay.Constant][1] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 112, 112), float32] */;
  %2 = nn.relu(%1) /* ty=Tensor[(1, 64, 112, 112), float32] span=aten__relu__0:0:0 */;
  %3 = nn.max_pool2d(%2, pool_size=[3, 3], strides=[2, 2], padding=[1, 1, 1, 1]) /* ty=Tensor[(1, 64, 56, 56), float32] span=aten__max_pool2d_0:0:0 */;
  %4 = nn.conv2d(%3, meta[relay.Constant][2] /* ty=Tensor[(64, 64, 3, 3), float32] */, padding=[1, 1, 1, 1], channels=64, kernel_size=[3, 3]) /* ty=Tensor[(1, 64, 56, 56), float32] */;
  %5 = add(%4, meta[relay.Constant][3] /* ty=Tensor[(64, 1, 1), float32] */) /* ty=Tensor[(1, 64, 56, 